In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Hyperparameters
# -----------------------------
input_size = 10
hidden_size = 32
sequence_length = 5
batch_size = 16
num_classes = 2
epochs = 20
dataset_size = 256  # more than one batch so training is meaningful

# -----------------------------
# Create Dummy Dataset
# -----------------------------
X = np.random.randn(dataset_size, sequence_length, input_size).astype("float32")
y = np.random.randint(0, 2, size=(dataset_size,)).astype("int64")

# Convert to tensors (keep them on CPU; move to device inside loop)
X_tensor = torch.tensor(X)
y_tensor = torch.tensor(y)

dataset = TensorDataset(X_tensor, y_tensor)

# Train / validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

# -----------------------------
# GRU Cell From Scratch
# -----------------------------
class GRUCellScratch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Update Gate
        self.Wz = nn.Linear(input_size, hidden_size)
        self.Uz = nn.Linear(hidden_size, hidden_size)
        # Reset Gate
        self.Wr = nn.Linear(input_size, hidden_size)
        self.Ur = nn.Linear(hidden_size, hidden_size)
        # Candidate Hidden State
        self.Wh = nn.Linear(input_size, hidden_size)
        self.Uh = nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h_prev):
        z_t = torch.sigmoid(self.Wz(x) + self.Uz(h_prev))
        r_t = torch.sigmoid(self.Wr(x) + self.Ur(h_prev))
        h_tilde = torch.tanh(self.Wh(x) + self.Uh(r_t * h_prev))
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        return h_t

# -----------------------------
# GRU Network
# -----------------------------
class GRUNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.gru_cell = GRUCellScratch(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        for t in range(x.size(1)):
            x_t = x[:, t, :]
            h = self.gru_cell(x_t, h)
        out = self.fc(h)
        return out

# -----------------------------
# Initialize Model, Loss, Optimizer
# -----------------------------
model = GRUNetwork(input_size, hidden_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# Training Loop with Validation
# -----------------------------
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == batch_y).sum().item()
        train_total += batch_x.size(0)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += batch_x.size(0)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss/train_total:.4f} Train Acc: {train_correct/train_total:.3f} "
        f"Val Loss: {val_loss/val_total:.4f} Val Acc: {val_correct/val_total:.3f}"
    )

# -----------------------------
# Final Predictions on Validation Set
# -----------------------------
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(batch_y)

predictions = torch.cat(all_preds)
labels = torch.cat(all_labels)
print("\nValidation Predictions:")
print(predictions)
print("Validation Labels:")
print(labels)


Using device: cpu
Epoch [1/20] Train Loss: 0.7013 Train Acc: 0.495 Val Loss: 0.7014 Val Acc: 0.538
Epoch [2/20] Train Loss: 0.6910 Train Acc: 0.505 Val Loss: 0.6986 Val Acc: 0.538
Epoch [3/20] Train Loss: 0.6847 Train Acc: 0.534 Val Loss: 0.6960 Val Acc: 0.558
Epoch [4/20] Train Loss: 0.6782 Train Acc: 0.564 Val Loss: 0.6946 Val Acc: 0.558
Epoch [5/20] Train Loss: 0.6729 Train Acc: 0.574 Val Loss: 0.6914 Val Acc: 0.596
Epoch [6/20] Train Loss: 0.6667 Train Acc: 0.598 Val Loss: 0.6901 Val Acc: 0.615
Epoch [7/20] Train Loss: 0.6615 Train Acc: 0.623 Val Loss: 0.6883 Val Acc: 0.635
Epoch [8/20] Train Loss: 0.6553 Train Acc: 0.642 Val Loss: 0.6876 Val Acc: 0.635
Epoch [9/20] Train Loss: 0.6486 Train Acc: 0.642 Val Loss: 0.6855 Val Acc: 0.635
Epoch [10/20] Train Loss: 0.6424 Train Acc: 0.662 Val Loss: 0.6828 Val Acc: 0.654
Epoch [11/20] Train Loss: 0.6351 Train Acc: 0.686 Val Loss: 0.6818 Val Acc: 0.635
Epoch [12/20] Train Loss: 0.6270 Train Acc: 0.691 Val Loss: 0.6814 Val Acc: 0.635
Epoch [